In [ ]:
!pip install openai
!pip install bert-score
!pip install transformers
!pip install huggingface
!pip install dataset
!pip install accelerate
!pip install --upgrade typing_extensions
!pip install openai
!pip install --upgrade transformers
!pip install datasets
!pip install sentence-transformers
!pip install scikit-learn
!pip install bert-score rouge-score nltk
!pip install tqdm
!pip install -q transformers datasets detoxify accelerate

In [ ]:
from huggingface_hub import login
login(token = "hf_xxxxxxxxxxxxx")

In [ ]:
import csv
import os
import json

def save_results_to_csv(result, features, file_path):
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        row = [result.get(feature, "") for feature in features]
        writer.writerow(row)


def converting_from_csv_to_json(features, csv_path, json_path):
    results = []
    with open(csv_path, 'r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)

        for row in csv_reader:

            result = {feature: value for feature, value in zip(features, row)}
            results.append(result)

    with open(json_path, 'w', encoding='utf-8') as json_file:
        json.dump(results, json_file, indent=4, ensure_ascii=False)

## COT BASELINE - GSM8K

In [ ]:
def plot_metrics(bert_scores, bleu_scores, rouge_scores, save_path):
    plt.figure(figsize=(10, 6))
    plt.plot(bert_scores, label="BERTScore F1")
    plt.plot(bleu_scores, label="BLEU")
    plt.plot(rouge_scores, label="ROUGE-L F")
    plt.xlabel("Sample Index")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics for Refined Answers")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()

In [ ]:
def plot_toxicities(expert_toxicities, amateur_toxicities, refined_toxicities, save_path):
    plt.figure(figsize=(12, 6))
    x = range(1, len(expert_toxicities) + 1)

    plt.plot(x, expert_toxicities, label="Expert Feedback Toxicity", color="blue", marker="o")
    plt.plot(x, amateur_toxicities, label="Amateur Feedback Toxicity", color="green", marker="^")
    plt.plot(x, refined_toxicities, label="Refined Answer Toxicity", color="red", marker="s")

    plt.xlabel("Task Index")
    plt.ylabel("Toxicity Score")
    plt.title("Toxicity Scores per Task")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Toxicity plot saved to: {save_path}")

In [ ]:
import random
import torch
import numpy as np
from datasets import load_dataset
# from torch.utils.data import DataLoader

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

def generate_dataset(dataset_name, if_gsm8k=False):
    # Load dataset
    if if_gsm8k:
        dataset = load_dataset(dataset_name, 'main')
    else:
        dataset = load_dataset(dataset_name)

    train_dataset = dataset['train']

    if 'test' in dataset:
        test_dataset = dataset['test']
    else:
        # Manually split train set into 90/10 for train/test
        split = train_dataset.train_test_split(test_size=0.1, seed=42)
        train_dataset = split['train']
        test_dataset = split['test']

    # Select 5 random samples for debug set
    rand_indices = [random.randint(0, len(train_dataset) - 1) for _ in range(5)]
    debug_dataset = train_dataset.select(rand_indices)

    return train_dataset, test_dataset, debug_dataset

In [ ]:
from datasets import load_dataset

def loading_dataset(dataset_list, dataset_id_list):
    all_datasets = []

    for i in range(len(dataset_list)):
        dataset_name = dataset_list[i]
        dataset_id = dataset_id_list[i]
        is_gsm8k = dataset_name.lower() == "gsm8k"

        train_dataset, test_dataset, debug_dataset = generate_dataset(dataset_id, is_gsm8k)

        all_datasets.append((train_dataset, test_dataset, debug_dataset))

    return all_datasets

In [ ]:
dataset_list = ["gsm8k", "alpaca"]
dataset_id = ["openai/gsm8k", "tatsu-lab/alpaca"]

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import re


def extract_final_score_and_feedback(evaluation_text):
    # Extract all scores (numbers) without trailing dots
    scores = re.findall(r"Final Score:\s*([-\d]+(?:\.\d+)?)", evaluation_text)

    # Extract all feedback blocks after 'Final Feedback:',
    # stopping before next Final Score:, Final Feedback:, or end of string
    feedbacks = re.findall(r"Final Feedback:\s*(.*?)(?=Final Score:|Final Feedback:|$)", evaluation_text, re.S)

    print(f"Scores found: {scores}")
    print(f"Feedbacks found: {len(feedbacks)}")

    final_score = float(scores[-1]) if scores else None
    final_feedback = feedbacks[-1].strip() if feedbacks else ""

    return final_score, final_feedback

def truncate_to_first_paragraph(text):
    for part in text.strip().split("\n\n"):
        if part.strip():
            return part.strip()
    return text.strip()

class ChainOfThoughtModel:
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_bf16=False):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_dtype = (
            torch.bfloat16 if use_bf16 else
            (torch.float16 if self.device == "cuda" else torch.float32)
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map="auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens=300, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.5,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_feedback_with_cot(self, prompt, answer):
        full_prompt = f"""
            You are an expert evaluator. Follow these steps:
            Step 1: Identify the task's key requirements, ideal response type, and evaluation criteria.
            Step 2: Check if the answer meets requirements, is factually accurate, and free of reasoning gaps.
            Step 3: Assess argument structure, clarity, evidence, and strengths/weaknesses.
            Step 4: Identify missing points, weak reasoning, unclear sections, and misconceptions.
            Step 5: Suggest changes to retain strengths, fix weaknesses, and improve clarity.
            Think carefully through each step to ensure your final assessment is thorough and helpful.
            Step 1: <one-paragraph>
            Step 2: <one paragraph>
            Step 3: <one paragraph>
            Step 4: <one paragraph>
            Step 5: <one paragraph>

            Final Score: <number>
            Final Feedback: <one paragraph>

            Question: Explain how the author uses symbolism to show the protagonist's emotional journey.
            Answer: The author uses the broken mirror as a symbol for the protagonist’s emotional state. At first, the mirror is whole, representing clarity and stability, but as the story progresses, it cracks and becomes distorted, reflecting the protagonist's growing inner turmoil and confusion.

            Step 1: The task requires identifying and explaining how symbolism illustrates the protagonist’s emotional journey. The ideal response should clearly connect symbols to emotional states and provide interpretive insights. Evaluation criteria include clarity, accuracy, relevance, and depth of analysis.

            Step 2: The answer directly addresses the prompt by focusing on the broken mirror symbol. The factual content about the mirror’s state progression is accurate and relevant. There are no apparent logical gaps.

            Step 3: The argument is clear and coherent, with a logical structure that moves from the symbol’s physical state to its emotional significance. The evidence (mirror's condition over time) is appropriate. Strengths include focused analysis and relevant symbolism. A weakness is a lack of specific textual examples or quotes to support the interpretation.

            Step 4: The answer could be improved by mentioning other symbols or elaborating on how the mirror affects the protagonist’s actions or feelings. It might also clarify the emotional nuances more deeply. There are no misconceptions detected.

            Step 5: To improve, retain the focus on the mirror symbol but add specific references to the text to support claims. Enhance emotional analysis by linking the symbolism more explicitly to the protagonist’s psychological changes. Clarify transitions to strengthen the flow.

            Final Score: 0.8.
            Final Feedback: Good. The answer is thoughtful and relevant but would benefit from more detailed textual support and deeper emotional exploration.

            Question: {prompt}
            Answer: {answer}
            """

        feedback = self.generate_text(full_prompt, max_new_tokens=500)

        # Extract last final score and feedback
        final_score, final_feedback = extract_final_score_and_feedback(feedback)

        return {
            "final_score": final_score,
            "final_feedback": truncate_to_first_paragraph(final_feedback)
        }

In [ ]:
cot_model = ChainOfThoughtModel()

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def truncate_to_first_paragraph(text):
    for part in text.strip().split("\n\n"):
        if part.strip():
            return part.strip()
    return text.strip()

class BaseModel:
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_bf16=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_dtype = (
            torch.bfloat16 if use_bf16 else
            torch.float16 if self.device == "cuda" else torch.float32
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map="auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=temperature,
                top_p=0.5,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, task_description=None, max_new_tokens=300):
        question = question.strip()
        task_description = task_description.strip() if task_description else None

        if task_description:
            prompt = (
                "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
                "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone.\n"
                "Write in a single paragraph, no lists or bullet points.\n"
                f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
                f"Task Description: {task_description}\n"
                f"User Input: {question}\n\n"
                "Answer:"
            )
        else:
            prompt = (
                "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
                "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone.\n"
                "Write in a single paragraph, no lists or bullet points.\n"
                f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
                f"Question: {question}\n\n"
                "Answer:"
            )

        response = self.generate_text(prompt, max_new_tokens)
        match = re.search(r"Answer:\s*(.+)", response, re.S)
        answer = match.group(1).strip() if match else response.strip()

        return truncate_to_first_paragraph(answer)

    def apply_feedback(self, question, answer, feedback, max_new_tokens=300):
        prompt = (
            "You are an expert language model tasked with producing precise, well-reasoned, and informative responses.\n"
            "Your goal is to revise the original answer using the provided feedback to create a response that is clear, accurate, and accessible.\n"
            "Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a reliable, logically structured, and self-contained explanation.\n"
            "Your revised response will help readers develop a foundational understanding of the topic.\n"
            "When crafting the response: 1. Identify the core message. 2. Present it in a logically structured, accurate, and easy-to-understand manner.\n"
            "Maintain a professional and concise tone. Avoid jargon, filler words, and emojis.\n"
            "Revise the following answer using the feedback provided. Ensure the final version is accurate, concise, and self-contained. Do not include any headings or meta-commentary.\n"
            f"Limit the revision to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
            f"Question:\n{question}\n\n"
            f"Original Answer:\n{answer}\n\n"
            f"Feedback:\n{feedback}\n\n"
            "Revised Answer:"
        )

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens)
        answer = response.split("Revised Answer:", 1)[-1].strip()
        return truncate_to_first_paragraph(answer)

In [ ]:
base_model = BaseModel()

In [ ]:
from datasets import load_dataset

# Load the test split of GSM8K
gsm8k_dataset = load_dataset("openai/gsm8k", "main")  # 'main' is the clean version
gsm8k = gsm8k_dataset["test"]  # or "train" depending on what you want

In [ ]:
def extract_final_answer(text):
    # Try to find any number in the text, optionally preceded by hashes or 'Answer:'
    match = re.search(r"(?:####\s*)?(-?\d+(?:\.\d+)?)", text)
    if not match:
        return None
    try:
        return float(match.group(1)) if '.' in match.group(1) else int(match.group(1))
    except ValueError:
        return None

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(accuracies, model_name):
    plt.figure(figsize=(10, 6))
    running_avg = [sum(accuracies[:i+1])/(i+1) for i in range(len(accuracies))]
    plt.plot(range(1, len(accuracies)+1), running_avg, marker='x', linestyle='--', label='Running Average Accuracy')

    plt.xlabel('Sample Number')
    plt.ylabel('Accuracy')
    plt.title(f"Model Accuracy on GSM8K Samples using {model_name}")
    plt.ylim(-0.1, 1.1)
    plt.xticks(range(1, len(accuracies)+1))
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
    import os
    from detoxify import Detoxify

    def run_experiment_gsm8k_cot(dataset, dataset_name, cot_model, base_model, csv_dir="./results", task_description="Solve the following math problem step-by-step"):
        os.makedirs(csv_dir, exist_ok=True)
        csv_path = f"{csv_dir}/{dataset_name}_experimental_results_cot_gsm8k.csv"
        json_path = f"{csv_dir}/{dataset_name}_experimental_results_cot_gsm8k.json"

        features = ["question", "ground_truth", "initial_answer", "final_answer", "accuracy"]

        results = []
        accuracies = []

        for i, sample in enumerate(dataset):
            if i >= 500:
                break


            input_prompt = sample["question"]
            ground_truth = extract_final_answer(sample["answer"])

            initial_output = base_model.generate_answer(input_prompt)
            # Fix here: pass correct variables
            cot_feedback = cot_model.generate_feedback_with_cot(input_prompt, initial_output)
            final_output = base_model.apply_feedback(input_prompt, initial_output, cot_feedback)
            predicted_answer = extract_final_answer(final_output)
            accuracy = ground_truth == predicted_answer

            result = {
                "question": input_prompt,
                "ground_truth": ground_truth,
                "initial_answer": initial_output,
                "final_answer": final_output,
                "accuracy": accuracy
            }
            results.append(result)
            accuracies.append(accuracy)
            save_results_to_csv(result, features, csv_path)

        converting_from_csv_to_json(features, csv_path, json_path)
        avg_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0
        print(f"Average accuracy over {len(accuracies)} samples: {avg_accuracy:.2%}")

        plot_accuracy(accuracies, "cot_model")
        return results, avg_accuracy

In [ ]:
gsm8k_experiment_result_cot = run_experiment_gsm8k_cot(gsm8k, "gsm8k", cot_model, base_model,"./results", "Solve the following math problem step-by-step")

## COD Baseline - GSM8K

In [ ]:
def plot_metrics(bert_scores, bleu_scores, rouge_scores, save_path):
    plt.figure(figsize=(10, 6))
    plt.plot(bert_scores, label="BERTScore F1")
    plt.plot(bleu_scores, label="BLEU")
    plt.plot(rouge_scores, label="ROUGE-L F")
    plt.xlabel("Sample Index")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics for Refined Answers")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()

In [ ]:
def plot_toxicities(expert_toxicities, amateur_toxicities, refined_toxicities, save_path):
    plt.figure(figsize=(12, 6))
    x = range(1, len(expert_toxicities) + 1)

    plt.plot(x, expert_toxicities, label="Expert Feedback Toxicity", color="blue", marker="o")
    plt.plot(x, amateur_toxicities, label="Amateur Feedback Toxicity", color="green", marker="^")
    plt.plot(x, refined_toxicities, label="Refined Answer Toxicity", color="red", marker="s")

    plt.xlabel("Task Index")
    plt.ylabel("Toxicity Score")
    plt.title("Toxicity Scores per Task")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Toxicity plot saved to: {save_path}")

In [ ]:
import random
import torch
import numpy as np
from datasets import load_dataset
# from torch.utils.data import DataLoader

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

def generate_dataset(dataset_name, if_gsm8k=False):
    # Load dataset
    if if_gsm8k:
        dataset = load_dataset(dataset_name, 'main')
    else:
        dataset = load_dataset(dataset_name)

    train_dataset = dataset['train']

    if 'test' in dataset:
        test_dataset = dataset['test']
    else:
        # Manually split train set into 90/10 for train/test
        split = train_dataset.train_test_split(test_size=0.1, seed=42)
        train_dataset = split['train']
        test_dataset = split['test']

    # Select 5 random samples for debug set
    rand_indices = [random.randint(0, len(train_dataset) - 1) for _ in range(5)]
    debug_dataset = train_dataset.select(rand_indices)

    return train_dataset, test_dataset, debug_dataset

In [ ]:
from datasets import load_dataset

def loading_dataset(dataset_list, dataset_id_list):
    all_datasets = []

    for i in range(len(dataset_list)):
        dataset_name = dataset_list[i]
        dataset_id = dataset_id_list[i]
        is_gsm8k = dataset_name.lower() == "gsm8k"

        train_dataset, test_dataset, debug_dataset = generate_dataset(dataset_id, is_gsm8k)

        all_datasets.append((train_dataset, test_dataset, debug_dataset))

    return all_datasets

In [ ]:
dataset_list = ["gsm8k", "alpaca"]
dataset_id = ["openai/gsm8k", "tatsu-lab/alpaca"]

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
def truncate_to_first_paragraph(text):
    for part in text.strip().split("\n\n"):
        if part.strip():
            return part.strip()
    return text.strip()

class ChainOfDraftModel:
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_bf16=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_dtype = (
            torch.bfloat16 if use_bf16 else
            (torch.float16 if self.device == "cuda" else torch.float32)
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map="auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens=300, temperature=0.7):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.5,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_feedback(self, question, answer):
        prompt = (
            "You are an expert feedback evaluator using Chain of Draft methodology. Your task is to iteratively develop "
            "a comprehensive assessment of the quality and relevance of the answer provided below.\n"
            "The answer reflects on subjective content, such as literature, personal interpretation, or emotional response.\n\n"

            "Draft 1 Assessment:\n"
            "Quickly analyze the answer and provide:\n"
            "- A preliminary score based on your first impression\n"
            "- Brief identification of the most obvious strength\n"
            "- The most apparent weakness or area for improvement\n"
            "Keep this assessment concise and direct.\n\n"

            "Draft 2 Analysis:\n"
            "Review your Draft 1 and expand your evaluation by:\n"
            "- Reconsidering your initial score with deeper analysis\n"
            "- Identifying additional strengths and weaknesses\n"
            "- Analyzing the answer's structure, insight depth, and emotional engagement\n"
            "- Considering textual support and interpretive quality\n"
            "- Refining your score if necessary\n\n"

            "Draft 3 Final Assessment:\n"
            "Integrate insights from previous drafts to provide:\n"
            "- Your final, carefully considered score\n"
            "- A polished, constructive paragraph of feedback\n"
            "- Professional, concise, and objective tone\n"
            "- Specific suggestions for improvement\n"
            "- No personal encouragements or vague generalities\n\n"

            "Use the following scoring scale consistently across all drafts:\n"
            "  •  1.0 — Excellent: insightful, emotionally engaging, and well-structured\n"
            "  •  0.7–0.9 — Good: mostly clear and thoughtful with minor issues\n"
            "  •  0.4–0.6 — Fair: has substance but lacks clarity or cohesion\n"
            "  •  0.1–0.3 — Weak: vague, underdeveloped, or emotionally flat\n"
            "  •  0.0 — Not helpful: irrelevant or incoherent\n"
            "  • -0.1 to -0.4 — Somewhat misleading or confusing\n"
            "  • -0.5 to -0.9 — Mostly incorrect, misrepresents the source\n"
            "  • -1.0 — Harmful or dangerously misleading\n\n"

            "Think carefully through each draft to ensure your final assessment is thorough and helpful.\n\n"
            "Draft 1 Assessment:\n"
            "Score: <number>\n"
            "Initial thoughts: <brief assessment>\n\n"
            "Draft 2 Analysis:\n"
            "Score: <number>\n"
            "Deeper analysis: <expanded evaluation>\n\n"
            "Draft 3 Final Assessment:\n"
            "Score: <number>\n"
            "Feedback: <polished paragraph>"

            "Question: How does the author use symbolism in the story to convey the protagonist's emotional journey?\n"
            "Answer: The author uses the image of the broken mirror to show how the protagonist feels broken inside. "
            "Every time the mirror is mentioned, it reflects a different stage of the character's emotional turmoil. "
            "At the beginning, the mirror is clean and whole, but as the story progresses, it becomes cracked and dusty. "
            "This mirrors the character's descent into depression and self-doubt. "
            "I think the mirror also shows how the character sees themselves — not clearly, but distorted through their pain.\n\n"

            "Draft 1 Assessment:\n"
            "Score: 0.8\n"
            "Initial thoughts: Strong symbolic identification and progression tracking. Needs more textual evidence.\n\n"

            "Draft 2 Analysis:\n"
            "Score: 0.85\n"
            "Deeper analysis: Good symbolic progression, thoughtful interpretation of distortion metaphor, but informal tone in places. Structure is solid overall.\n\n"

            "Draft 3 Final Assessment:\n"
            "Score: 0.85\n"
            "Feedback: This answer provides a thoughtful and coherent interpretation of the mirror as a symbol of the protagonist's emotional state. "
            "The symbolic progression is well articulated, and the reflection on distortion is particularly effective. "
            "To improve, the student could enhance their analysis by citing specific textual moments or imagery. "
            "Overall, this is a strong and emotionally attuned response.\n\n"

            "Now evaluate the following:\n\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
        )

        feedback = self.generate_text(prompt).strip()

        # Extract last occurrence of each draft using findall:
        draft1_matches = re.findall(
            r"Draft 1 Assessment:\s*Score:\s*([-\d\.]+)\s*Initial thoughts:\s*(.+?)(?=\n\nDraft 2|$)",
            feedback,
            re.S
        )
        draft2_matches = re.findall(
            r"Draft 2 Analysis:\s*Score:\s*([-\d\.]+)\s*Deeper analysis:\s*(.+?)(?=\n\nDraft 3|$)",
            feedback,
            re.S
        )
        draft3_matches = re.findall(
            r"Draft 3 Final Assessment:\s*Score:\s*([-\d\.]+)\s*Feedback:\s*(.+)",
            feedback,
            re.S
        )

        draft1 = draft1_matches[-1] if draft1_matches else (None, "")
        draft2 = draft2_matches[-1] if draft2_matches else (None, "")
        draft3 = draft3_matches[-1] if draft3_matches else (None, "")

        return {
            "draft1": {
                "score": float(draft1[0]) if draft1[0] else None,
                "thoughts": truncate_to_first_paragraph(draft1[1].strip())
            },
            "draft2": {
                "score": float(draft2[0]) if draft2[0] else None,
                "analysis": truncate_to_first_paragraph(draft2[1].strip())
            },
            "draft3": {
                "score": float(draft3[0]) if draft3[0] else None,
                "feedback": truncate_to_first_paragraph(draft3[1].strip())
            }
        }

In [ ]:
cod_model = ChainOfDraftModel()

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def truncate_to_first_paragraph(text):
    for part in text.strip().split("\n\n"):
        if part.strip():
            return part.strip()
    return text.strip()

class BaseModel:
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_bf16=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_dtype = (
            torch.bfloat16 if use_bf16 else
            torch.float16 if self.device == "cuda" else torch.float32
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map="auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=temperature,
                top_p=0.5,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, task_description=None, max_new_tokens=300):
        question = question.strip()
        task_description = task_description.strip() if task_description else None

        if task_description:
            prompt = (
                "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
                "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone.\n"
                "Write in a single paragraph, no lists or bullet points.\n"
                f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
                f"Task Description: {task_description}\n"
                f"User Input: {question}\n\n"
                "Answer:"
            )
        else:
            prompt = (
                "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
                "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone.\n"
                "Write in a single paragraph, no lists or bullet points.\n"
                f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
                f"Question: {question}\n\n"
                "Answer:"
            )

        response = self.generate_text(prompt, max_new_tokens)
        match = re.search(r"Answer:\s*(.+)", response, re.S)
        answer = match.group(1).strip() if match else response.strip()

        return truncate_to_first_paragraph(answer)

    def apply_feedback(self, question, answer, feedback, max_new_tokens=300):
        prompt = (
            "You are an expert language model tasked with producing precise, well-reasoned, and informative responses.\n"
            "Your goal is to revise the original answer using the provided feedback to create a response that is clear, accurate, and accessible.\n"
            "Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a reliable, logically structured, and self-contained explanation.\n"
            "Your revised response will help readers develop a foundational understanding of the topic.\n"
            "When crafting the response: 1. Identify the core message. 2. Present it in a logically structured, accurate, and easy-to-understand manner.\n"
            "Maintain a professional and concise tone. Avoid jargon, filler words, and emojis.\n"
            "Revise the following answer using the feedback provided. Ensure the final version is accurate, concise, and self-contained. Do not include any headings or meta-commentary.\n"
            f"Limit the revision to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
            f"Question:\n{question}\n\n"
            f"Original Answer:\n{answer}\n\n"
            f"Feedback:\n{feedback}\n\n"
            "Revised Answer:"
        )

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens)
        answer = response.split("Revised Answer:", 1)[-1].strip()
        return truncate_to_first_paragraph(answer)

In [ ]:
base_model = BaseModel()

In [ ]:
def extract_final_answer(text):
    # Try to find any number in the text, optionally preceded by hashes or 'Answer:'
    match = re.search(r"(?:####\s*)?(-?\d+(?:\.\d+)?)", text)
    if not match:
        return None
    try:
        return float(match.group(1)) if '.' in match.group(1) else int(match.group(1))
    except ValueError:
        return None

In [ ]:
from datasets import load_dataset

# Load the test split of GSM8K
gsm8k_dataset = load_dataset("openai/gsm8k", "main")  # 'main' is the clean version
gsm8k = gsm8k_dataset["test"]  # or "train" depending on what you want

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(accuracies, model_name):
    plt.figure(figsize=(10, 6))
    running_avg = [sum(accuracies[:i+1])/(i+1) for i in range(len(accuracies))]
    plt.plot(range(1, len(accuracies)+1), running_avg, marker='x', linestyle='--', label='Running Average Accuracy')

    plt.xlabel('Sample Number')
    plt.ylabel('Accuracy')
    plt.title(f"Model Accuracy on GSM8K Samples using {model_name}")
    plt.ylim(-0.1, 1.1)
    plt.xticks(range(1, len(accuracies)+1))
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
def run_experiment_gsm8k_cod(dataset, dataset_name, cod_model, base_model, csv_dir="./results", task_description="Solve the following math problem step-by-step"):
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f"{csv_dir}/{dataset_name}_experimental_results_cod_gsm8k.csv"
    json_path = f"{csv_dir}/{dataset_name}_experimental_results_cod_gsm8k.json"

    features = ["question", "ground_truth", "initial_answer", "cod_feedback", "final_answer", "accuracy"]

    results = []
    accuracies = []

    for i, sample in enumerate(dataset):
        if i >= 500:
            break

        input_prompt = sample["question"]
        ground_truth = extract_final_answer(sample["answer"])

        initial_output = base_model.generate_answer(input_prompt, task_description=task_description)
        cod_feedback = cod_model.generate_feedback(input_prompt, initial_output)
        final_output = base_model.apply_feedback(input_prompt, initial_output, cod_feedback)

        # Extract numeric answers from outputs
        final_answer_num = extract_final_answer(final_output)

        predicted_answer = extract_final_answer(final_output)
        predicted_answer = extract_final_answer(final_output)
        accuracy = int(ground_truth == predicted_answer)


        result = {
            "question": input_prompt,
            "ground_truth": ground_truth,
            "initial_answer": initial_output,
            "cod_feedback": cod_feedback,
            "final_answer": final_output,
            "accuracy": accuracy
        }

        results.append(result)
        accuracies.append(accuracy)

        save_results_to_csv(result, features, csv_path)

    converting_from_csv_to_json(features, csv_path, json_path)

    avg_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0
    print(f"Average accuracy over {len(accuracies)} samples: {avg_accuracy:.2%}")

    plot_accuracy(accuracies, "CoD Model")

    return results, avg_accuracy


In [ ]:
gsm8k_experiment_result_cod = run_experiment_gsm8k_cod(gsm8k, "gsm8k", cod_model, base_model, "./results", "Solve the following math problem step-by-step")

## CLEAR Baselines - GSM8K

In [ ]:
import random
import torch
import numpy as np
from datasets import load_dataset
# from torch.utils.data import DataLoader

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

def generate_dataset(dataset_name, if_gsm8k=False):
    # Load dataset
    if if_gsm8k:
        dataset = load_dataset(dataset_name, 'main')
    else:
        dataset = load_dataset(dataset_name)

    train_dataset = dataset['train']

    if 'test' in dataset:
        test_dataset = dataset['test']
    else:
        # Manually split train set into 90/10 for train/test
        split = train_dataset.train_test_split(test_size=0.1, seed=42)
        train_dataset = split['train']
        test_dataset = split['test']

    # Select 5 random samples for debug set
    rand_indices = [random.randint(0, len(train_dataset) - 1) for _ in range(5)]
    debug_dataset = train_dataset.select(rand_indices)

    return train_dataset, test_dataset, debug_dataset

In [ ]:
from datasets import load_dataset

def loading_dataset(dataset_list, dataset_id_list):
    all_datasets = []

    for i in range(len(dataset_list)):
        dataset_name = dataset_list[i]
        dataset_id = dataset_id_list[i]
        is_gsm8k = dataset_name.lower() == "gsm8k"

        train_dataset, test_dataset, debug_dataset = generate_dataset(dataset_id, is_gsm8k)

        all_datasets.append((train_dataset, test_dataset, debug_dataset))

    return all_datasets

In [ ]:
dataset_list = ["gsm8k", "alpaca"]
dataset_id = ["openai/gsm8k", "tatsu-lab/alpaca"]

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def truncate_at_bracket(output: str) -> str:
    return output.split("[")[0].strip()

def truncate_to_first_paragraph(text):
    for part in text.strip().split('\n\n'):
        paragraph = part.strip().split('\n')[0].strip()
        if paragraph:
            return paragraph
    return text.strip()

class ExpertFeedbackModel:
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_bf16=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_dtype = torch.bfloat16 if use_bf16 else torch.float16 if self.device == "cuda" else torch.float32

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        # Load model with correct dtype and device
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map = "auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens=300, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=temperature,
                top_p=0.5,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, task_description=None, max_new_tokens=300, max_words=100):
      """
      Generates a concise answer based on the given question and optional task description.
      Uses different prompt templates depending on whether task_description is provided.
      """

      question = question.strip()
      task_description = task_description.strip() if task_description else None
      if task_description:
          prompt = (
            "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
            "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a reliable, self-contained answer that offers foundational understanding of the topic.\n"
            "When responding, follow this approach: 1) Identify the core message. 2) Present it in a logically structured, accurate, and easy-to-understand manner.\n"
            "Keep the tone professional and concise. Avoid jargon, filler, or emojis.\n"
            f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words. If a full explanation would be too long, summarize it while ensuring it remains self-contained and informative.\n"
            "Write in a single paragraph—no lists, bullet points, or follow-up suggestions.\n"
            "Think step by step to ensure clarity and alignment with the intended audience.\n\n"
            f"Task Description: {task_description}\n"
            f"User Input: {question}\n\n"
            "Answer:"
          )
      else:
          prompt = (
              "You are an expert language model tasked with providing precise, well-reasoned, and informative responses.\n"
              "Your goal is to deliver a clear, accessible, and accurate response in a formal yet approachable tone for a general audience with no assumed background knowledge.\n"
              "The user is an informed individual seeking a reliable, well-reasoned, and self-contained explanation.\n"
              "Your response will be used to inform readers seeking a foundational understanding of the topic.\n"
              "When responding to the request: 1. Identify the core message. 2. Explain it in a way that is logically structured, accurate, and easy to understand.\n"
              "Keep the tone professional and concise. Avoid jargon, filler words, and emojis.\n"
              f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words. If a full explanation would be too long, summarize it while ensuring it remains self-contained and informative.\n"
              "Write in a single paragraph with no lists, bullet points, or follow-up suggestions.\n"
              "Think step by step to ensure your response improves clarity and aligns with the target audience.\n\n"
              f"Question: {question}\n\n"
              "Answer:"
          )

      response = self.generate_text(prompt, max_new_tokens)
      answer = response.split("Answer:", 1)[-1].strip()
      answer = truncate_at_bracket(answer)

      return truncate_to_first_paragraph(answer)

    def generate_feedback(self, question, answer):
        prompt = (
            "You are an expert feedback evaluator. Your task is to assess the quality and relevance of the answer provided below.\n"
            "The answer reflects on subjective content, such as literature, personal interpretation, or emotional response.\n\n"
            "Follow these steps:\n"
            "1. Carefully analyze the answer. Internally identify its strengths, weaknesses, and areas for improvement.\n"
            "2. Write a clear, specific, and constructive paragraph of feedback that would help the user improve their response.\n"
            "3. Output your final **Score** and **Feedback** in the format shown below.\n\n"
            "Use a professional, concise, and objective tone. Do not include personal encouragements or vague generalities.\n"
            "Use the following scoring scale:\n"
            "  •  1.0 — Excellent: insightful, emotionally engaging, and well-structured\n"
            "  •  0.7–0.9 — Good: mostly clear and thoughtful with minor issues\n"
            "  •  0.4–0.6 — Fair: has substance but lacks clarity or cohesion\n"
            "  •  0.1–0.3 — Weak: vague, underdeveloped, or emotionally flat\n"
            "  •  0.0 — Not helpful: irrelevant or incoherent\n"
            "  • -0.1 to -0.4 — Somewhat misleading or confusing\n"
            "  • -0.5 to -0.9 — Mostly incorrect, misrepresents the source\n"
            "  • -1.0 — Harmful or dangerously misleading\n\n"
            "Think carefully before responding. Format your output as follows:\n\n"
            "Score: <number>\n"
            "Feedback: <one-paragraph response>\n\n"
            "Question: How does the author use symbolism in the story to convey the protagonist's emotional journey?\n"
            "Answer: The author uses the image of the broken mirror to show how the protagonist feels broken inside. "
            "Every time the mirror is mentioned, it reflects a different stage of the character’s emotional turmoil. "
            "At the beginning, the mirror is clean and whole, but as the story progresses, it becomes cracked and dusty. "
            "This mirrors the character’s descent into depression and self-doubt. "
            "I think the mirror also shows how the character sees themselves — not clearly, but distorted through their pain.\n\n"
            "Step-by-step reasoning: The response identifies a clear symbol and tracks its evolution through the story. "
            "It links the mirror’s changes to the protagonist’s internal state in a way that demonstrates thoughtful analysis. "
            "However, it could benefit from referencing specific moments or language from the text to make the argument stronger. "
            "Some phrasing is slightly informal, but the overall insight and structure are solid.\n\n"
            "Score: 0.85\n"
            "Feedback: This answer provides a thoughtful and coherent interpretation of the mirror as a symbol of the protagonist’s emotional state. "
            "The symbolic progression is well articulated, and the reflection on distortion is particularly effective. "
            "To improve, the student could enhance their analysis by citing specific textual moments or imagery. "
            "Overall, this is a strong and emotionally attuned response.\n\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
            "Score:"
        )

        response = self.generate_text(prompt, max_new_tokens=200).strip()
        # print(f"Response: {response}")

        # Extract all score values
        score_matches = re.findall(r"Score\s*:\s*(-?1(?:\.0)?|-?0(?:\.\d+)?|-\d\.\d+)", response)

        # Extract all feedback values (non-greedy multiline)
        feedback_matches = re.findall(r"Feedback\s*:\s*(.*?)(?=\n\S|$)", response, re.DOTALL)

        # Get the second score and feedback if available
        if len(score_matches) >= 2 and len(feedback_matches) >= 2:
            score = float(score_matches[1])
            feedback = feedback_matches[2].strip()
        else:
            score = 0.0
            feedback = "No feedback provided."

        return truncate_to_first_paragraph(feedback), score

    def prompt_condense_feedback(self, expert_feedback, amateur_feedback):
        return (
            "You are a feedback summarizer tasked with condensing two feedback comments on an answer.\n"
            "Your goal is to generate a clear and concise summary by prioritizing the expert feedback while incorporating any useful or original points from the amateur feedback.\n"
            "Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a balanced summary to improve their response.\n"
            "Your summary should provide informative and actionable feedback.\n"
            "In cases of disagreement, favor the expert’s insights and ensure the final version is coherent, constructive, and no more than 70 words.\n\n"
            "You are given two feedback comments on an answer:\n"
            f"- Expert Feedback (trusted): {expert_feedback.strip()}\n"
            f"- Amateur Feedback: {amateur_feedback.strip()}\n\n"
            "Your task is to write a unified feedback summary (max 70 words) that:\n"
            "- Prioritizes insights from the expert feedback\n"
            "- Incorporates any unique and constructive points from the amateur feedback\n"
            "- Resolves disagreements by favoring the expert\n"
            "- Uses contrastive reasoning to emphasize the strengths of the expert feedback\n\n"
            "Final Merged Feedback:"
        )

    def generate_unified_feedback(self, expert_feedback, amateur_feedback):
        if amateur_feedback == "Failed to generate meaningful feedback":
            return expert_feedback

        full_prompt = self.prompt_condense_feedback(expert_feedback, amateur_feedback)
        response = self.generate_text(full_prompt)

        if "Final Merged Feedback:" in response:
            return response.split("Final Merged Feedback:")[-1].strip()
        return response.strip()

In [ ]:
expert_model = ExpertFeedbackModel()

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from bert_score import score as bert_score  # optional
except ImportError:
    bert_score = None


class AmateurFeedbackModel:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0", use_bf16 = True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model_dtype = torch.bfloat16 if use_bf16 else torch.float16 if self.device == "cuda" else torch.float32
        # Load model with correct dtype and device
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map = "auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=2e-5)
        self.student_frozen = False

    def generate_text(self, prompt, max_new_tokens=300, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        model = self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=0.5,
            pad_token_id=self.tokenizer.eos_token_id
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_feedback(self, question, answer):
        prompt = (
            "You are an expert feedback evaluator. Your task is to assess the quality and relevance of the answer provided below.\n"
            "The answer reflects on subjective content, such as literature, personal interpretation, or emotional response.\n\n"
            "Follow these steps:\n"
            "1. Carefully analyze the answer. Internally identify its strengths, weaknesses, and areas for improvement.\n"
            "2. Write a clear, specific, and constructive paragraph of feedback that would help the user improve their response.\n"
            "3. Output your final **Score** and **Feedback** in the format shown below.\n\n"
            "Use a professional, concise, and objective tone. Do not include personal encouragements or vague generalities.\n"
            "Use the following scoring scale:\n"
            "  •  1.0 — Excellent: insightful, emotionally engaging, and well-structured\n"
            "  •  0.7–0.9 — Good: mostly clear and thoughtful with minor issues\n"
            "  •  0.4–0.6 — Fair: has substance but lacks clarity or cohesion\n"
            "  •  0.1–0.3 — Weak: vague, underdeveloped, or emotionally flat\n"
            "  •  0.0 — Not helpful: irrelevant or incoherent\n"
            "  • -0.1 to -0.4 — Somewhat misleading or confusing\n"
            "  • -0.5 to -0.9 — Mostly incorrect, misrepresents the source\n"
            "  • -1.0 — Harmful or dangerously misleading\n\n"
            "Think carefully before responding. Format your output as follows:\n\n"
            "Score: <number>\n"
            "Feedback: <one-paragraph response>\n\n"
            "Question: How does the author use symbolism in the story to convey the protagonist's emotional journey?\n"
            "Answer: The author uses the image of the broken mirror to show how the protagonist feels broken inside. "
            "Every time the mirror is mentioned, it reflects a different stage of the character’s emotional turmoil. "
            "At the beginning, the mirror is clean and whole, but as the story progresses, it becomes cracked and dusty. "
            "This mirrors the character’s descent into depression and self-doubt. "
            "I think the mirror also shows how the character sees themselves — not clearly, but distorted through their pain.\n\n"
            "Step-by-step reasoning: The response identifies a clear symbol and tracks its evolution through the story. "
            "It links the mirror’s changes to the protagonist’s internal state in a way that demonstrates thoughtful analysis. "
            "However, it could benefit from referencing specific moments or language from the text to make the argument stronger. "
            "Some phrasing is slightly informal, but the overall insight and structure are solid.\n\n"
            "Score: 0.85\n"
            "Feedback: This answer provides a thoughtful and coherent interpretation of the mirror as a symbol of the protagonist’s emotional state. "
            "The symbolic progression is well articulated, and the reflection on distortion is particularly effective. "
            "To improve, the student could enhance their analysis by citing specific textual moments or imagery. "
            "Overall, this is a strong and emotionally attuned response.\n\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
            "Score:"
        )

        response = self.generate_text(prompt, max_new_tokens=150).strip()
        # print(f"Response: {response}")

        # Extract all score values
        score_matches = re.findall(r"Score\s*:\s*(-?1(?:\.0)?|-?0(?:\.\d+)?|-\d\.\d+)", response)

        # Extract all feedback values (non-greedy multiline)
        feedback_matches = re.findall(r"Feedback\s*:\s*(.*?)(?=\n\S|$)", response, re.DOTALL)

        # Get the second score and feedback if available
        if len(score_matches) >= 2 and len(feedback_matches) >= 2:
            score = float(score_matches[1])
            feedback = feedback_matches[2].strip()
        else:
            score = 0.0
            feedback = "No feedback provided."

        return truncate_to_first_paragraph(feedback), score

In [ ]:
amateur_model = AmateurFeedbackModel()

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def truncate_to_first_paragraph(text):
    for part in text.strip().split("\n\n"):
        if part.strip():
            return part.strip()
    return text.strip()

class BaseModel:
    def __init__(self, model_name="meta-llama/Llama-3.1-8B-Instruct", use_bf16=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_dtype = (
            torch.bfloat16 if use_bf16 else
            torch.float16 if self.device == "cuda" else torch.float32
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=self.model_dtype,
            device_map="auto",
            trust_remote_code=True
        )

        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=temperature,
                top_p=0.5,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, task_description=None, max_new_tokens=300):
        question = question.strip()
        task_description = task_description.strip() if task_description else None

        if task_description:
            prompt = (
                "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
                "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone.\n"
                "Write in a single paragraph, no lists or bullet points.\n"
                f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
                f"Task Description: {task_description}\n"
                f"User Input: {question}\n\n"
                "Answer:"
            )
        else:
            prompt = (
                "You are an expert language model tasked with delivering precise, well-reasoned, and informative responses.\n"
                "Your goal is to provide a clear, accurate, and accessible explanation in a formal yet approachable tone.\n"
                "Write in a single paragraph, no lists or bullet points.\n"
                f"Limit your response to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
                f"Question: {question}\n\n"
                "Answer:"
            )

        response = self.generate_text(prompt, max_new_tokens)
        match = re.search(r"Answer:\s*(.+)", response, re.S)
        answer = match.group(1).strip() if match else response.strip()

        return truncate_to_first_paragraph(answer)

    def apply_feedback(self, question, answer, feedback, max_new_tokens=300):
        prompt = (
            "You are an expert language model tasked with producing precise, well-reasoned, and informative responses.\n"
            "Your goal is to revise the original answer using the provided feedback to create a response that is clear, accurate, and accessible.\n"
            "Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a reliable, logically structured, and self-contained explanation.\n"
            "Your revised response will help readers develop a foundational understanding of the topic.\n"
            "When crafting the response: 1. Identify the core message. 2. Present it in a logically structured, accurate, and easy-to-understand manner.\n"
            "Maintain a professional and concise tone. Avoid jargon, filler words, and emojis.\n"
            "Revise the following answer using the feedback provided. Ensure the final version is accurate, concise, and self-contained. Do not include any headings or meta-commentary.\n"
            f"Limit the revision to approximately {max_new_tokens // 1.3:.0f} words.\n\n"
            f"Question:\n{question}\n\n"
            f"Original Answer:\n{answer}\n\n"
            f"Feedback:\n{feedback}\n\n"
            "Revised Answer:"
        )

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens)
        answer = response.split("Revised Answer:", 1)[-1].strip()
        return truncate_to_first_paragraph(answer)

In [ ]:
base_model = BaseModel()

In [ ]:
def extract_final_answer(text):
    # Try to find any number in the text, optionally preceded by hashes or 'Answer:'
    match = re.search(r"(?:####\s*)?(-?\d+(?:\.\d+)?)", text)
    if not match:
        return None
    try:
        return float(match.group(1)) if '.' in match.group(1) else int(match.group(1))
    except ValueError:
        return None

In [ ]:
from datasets import load_dataset

# Load the test split of GSM8K
gsm8k_dataset = load_dataset("openai/gsm8k", "main")  # 'main' is the clean version
gsm8k = gsm8k_dataset["test"]  # or "train" depending on what you want

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(accuracies, model_name):
    plt.figure(figsize=(10, 6))
    running_avg = [sum(accuracies[:i+1])/(i+1) for i in range(len(accuracies))]
    plt.plot(range(1, len(accuracies)+1), running_avg, marker='x', linestyle='--', label='Running Average Accuracy')

    plt.xlabel('Sample Number')
    plt.ylabel('Accuracy')
    plt.title(f"Model Accuracy on GSM8K Samples using {model_name}")
    plt.ylim(-0.1, 1.1)
    plt.xticks(range(1, len(accuracies)+1))
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
import os

def run_experiment_gsm8k_clear(dataset, dataset_name, expert_model, amateur_model, base_model, csv_dir="./results", task_description="Solve the following math problem step-by-step"):
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f"{csv_dir}/{dataset_name}_experimental_results_clear_gsm8k.csv"
    json_path = f"{csv_dir}/{dataset_name}_experimental_results_clear_gsm8k.json"

    features = ["question", "ground_truth", "generated_answer", "amateur_feedback", "expert_feedback", "accuracy"]

    results = []
    accuracies = []

    for i, sample in enumerate(dataset):
        if i >= 500:
            break

        input_prompt = sample["question"]
        ground_truth = extract_final_answer(sample["answer"])

        initial_output = base_model.generate_answer(input_prompt, task_description=task_description)

        # Ensure feedback is always a string
        amateur_feedback = amateur_model.generate_feedback(input_prompt, initial_output)
        if isinstance(amateur_feedback, tuple):
            amateur_feedback = amateur_feedback[0]
        amateur_feedback = str(amateur_feedback)

        expert_feedback = expert_model.generate_feedback(input_prompt, initial_output)
        if isinstance(expert_feedback, tuple):
            expert_feedback = expert_feedback[0]
        expert_feedback = str(expert_feedback)

        unified_feedback = expert_model.generate_unified_feedback(expert_feedback, amateur_feedback)

        final_output = base_model.apply_feedback(input_prompt, initial_output, unified_feedback)

        predicted_answer = extract_final_answer(final_output)
        accuracy = int(ground_truth == predicted_answer)

        results.append({
            "question": input_prompt,
            "ground_truth": ground_truth,
            "generated_answer": final_output,
            "amateur_feedback": amateur_feedback,
            "expert_feedback": expert_feedback,
            "accuracy": accuracy
        })
        accuracies.append(accuracy)

        save_results_to_csv(results[-1], features, csv_path)

    converting_from_csv_to_json(features, csv_path, json_path)

    avg_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0
    print(f"Average accuracy over {len(accuracies)} samples: {avg_accuracy:.2%}")

    plot_accuracy(accuracies, "CLEAR Model")

    return results, avg_accuracy


In [ ]:
gsm8kCLEARresults = run_experiment_gsm8k_clear(gsm8k, "gsm8k", expert_model, amateur_model, base_model, "./results","Solve the following math problem step-by-step")